# NaturalFuzz — generated rows that look like real ones

A generated row is built out of parts of real ones: take a row from the corpus and
splice in the columns that decide a branch from a row that reaches it. Every value
genuinely occurred in that column — the right formats, plausible magnitudes — while the
*combination* is new, and aimed at coverage rather than at random.

In [ ]:
import os, sys, glob

ROOT = os.environ.get("BIGASTERISK_HOME") or os.path.abspath("..")

# Jars: a source checkout has them under modules/*/target, the Docker image under jars/.
JARS = sorted(glob.glob(f"{ROOT}/modules/*/target/scala-2.13/bigasterisk-*.jar")) \
    or sorted(glob.glob(f"{ROOT}/jars/bigasterisk-*.jar"))
if not JARS:
    raise SystemExit("No BigAsterisk jars found. Run: bin/sbt package")

FASTUTIL_JAR = os.environ.get("FASTUTIL_JAR") or next(iter(sorted(
    glob.glob(f"{ROOT}/jars/fastutil*.jar")
    + glob.glob(os.path.expanduser("~/Library/Caches/Coursier/**/fastutil-8.5.15.jar"), recursive=True)
    + glob.glob(os.path.expanduser("~/.cache/coursier/**/fastutil-8.5.15.jar"), recursive=True)
)), None)
if not FASTUTIL_JAR:
    raise SystemExit("fastutil jar not found. Run: bin/sbt package")

SPARK_JARS = ",".join(JARS + [FASTUTIL_JAR])
DATA = f"{ROOT}/examples/data"
sys.path.insert(0, f"{ROOT}/python")

## The data

Twelve orders across three customers. One of them, `o8`, is an outlier at
`99999` — every notebook here uses it as the thing to find.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import bigasterisk

spark = (bigasterisk.configure(SparkSession.builder)
    .master("local[2]")
    .appName("naturalfuzz-notebook")
    .config("spark.jars", SPARK_JARS)
    .config("spark.sql.adaptive.skewJoin.enabled", "false")
    .config("spark.ui.enabled", "false")
    .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

orders = spark.read.schema("oid STRING, cid STRING, amount INT").csv(f"{DATA}/orders.txt")
customers = spark.read.schema("cid STRING, name STRING").csv(f"{DATA}/customers.txt")
orders.createOrReplaceTempView("orders")
customers.createOrReplaceTempView("customers")

orders.show()

## The branch no single row reaches

Each row is first evaluated against every branch of the query, giving it one bit per
branch — its **path vector**. Rows with the same vector take the same path through the
query, so only a few of each are kept.

This query has two conjuncts, and no order satisfies both: the one large order belongs
to `c2`, and `c1`'s orders are all small.

In [ ]:
QUERY = "SELECT oid FROM orders WHERE amount > 90000 AND cid = 'c1'"

print("rows satisfying both:", spark.sql(QUERY).count())
orders.selectExpr("oid", "amount > 90000 AS big", "cid = 'c1' AS is_c1").show()

## Interleaving finds it

Reaching the conjunction means combining one row's `amount` with another's `cid`.
Splicing does that; drawing values for each column independently does not — an invented
string essentially never equals `'c1'`.

In [ ]:
fuzzer = bigasterisk.fuzz(spark)
seeds = {"orders": orders}

spliced = fuzzer.fuzz(QUERY, seeds, iterations=25, rows_per_table=6,
                      strategy="natural", seed=5)
drawn = fuzzer.fuzz(QUERY, seeds, iterations=25, rows_per_table=6,
                    strategy="random", seed=5)

CONJUNCTION = "((orders.amount > 90000) AND (orders.cid = 'c1'))"

for name, result in [("natural", spliced), ("random", drawn)]:
    print(f"{name:9} coverage {result.coverage:.2f}   conjunction reached:",
          CONJUNCTION in result.covered)
print()
for branch in sorted(spliced.covered):
    print(" ", branch)

## Check

In [ ]:
assert CONJUNCTION in spliced.covered, sorted(spliced.covered)
assert CONJUNCTION not in drawn.covered
assert spliced.coverage > drawn.coverage
assert spliced.failures == [], spliced.failures
print("OK")